# POI Recommendation System for Vietnam Destinations

This notebook is an end-to-end recommender organized as:

- Phần A — POI preprocessing: cleaning, lat/lon, rating, reviews, types, description, keywords, quality, and TF-IDF.
- Phần B — Synthetic user behavior: user_id, poi_id, timestamp, and action.
- Phần C — Behavioral recommender: popularity, association rules, and item-based CF.
- Phần D — Content model: TF-IDF + cosine for cold-start.
- Phần E — Hybrid: behavior for existing users and content for new users.
- Phần F — Contextual reranking: behavior, distance, quality, and type/context.
- Phần G — Fuzzy AHP + TOPSIS: criteria, pairwise matrix, CR, fuzzy weights, H = A x W, defuzzify, TOPSIS, and Top-K.
- Phần H — Evaluation: chronological train/test, HitRate@K, Recall@K, MRR@K, NDCG@K, coverage, and diversity.

The source workbook is the POI catalog. Synthetic interactions are deterministic development data and can later be replaced by real logs.


In [ ]:
from pathlib import Path
import math
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from rapidfuzz import fuzz
    RAPIDFUZZ_AVAILABLE = True
except ImportError:
    from difflib import SequenceMatcher
    RAPIDFUZZ_AVAILABLE = False

    class _FallbackFuzz:
        @staticmethod
        def ratio(a, b):
            return int(100 * SequenceMatcher(None, str(a), str(b)).ratio())

        @staticmethod
        def partial_ratio(a, b):
            a = str(a)
            b = str(b)
            if not a or not b:
                return 0
            shorter, longer = (a, b) if len(a) <= len(b) else (b, a)
            if shorter and shorter in longer:
                return 100
            return _FallbackFuzz.ratio(a, b)

        @staticmethod
        def token_set_ratio(a, b):
            a_tokens = set(str(a).split())
            b_tokens = set(str(b).split())
            if not a_tokens or not b_tokens:
                return 0
            shared = a_tokens & b_tokens
            union = a_tokens | b_tokens
            return int(100 * len(shared) / len(union))

    fuzz = _FallbackFuzz()

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)

DATA_PATH = Path('data/vietnam_destinations_google_maps_browser_hotosm.xlsx')
CLEAN_OUTPUT_PATH = Path('data/poi_recommendation_cleaned.xlsx')
SAMPLE_RECOMMENDATIONS_PATH = Path('data/poi_sample_recommendations.xlsx')
EVALUATION_OUTPUT_PATH = Path('data/poi_evaluation_metrics.xlsx')

print('RapidFuzz available:', RAPIDFUZZ_AVAILABLE)
print('Input exists:', DATA_PATH.exists(), DATA_PATH)

## Phần A — POI preprocessing

### A.1 Load and inspect the POI catalog

Inspect the scraped workbook before modeling so missing columns, invalid coordinates, and source-specific formats are visible.


In [ ]:
df_raw = pd.read_excel(DATA_PATH)

print('Shape:', df_raw.shape)
print('Columns:')
for idx, col in enumerate(df_raw.columns, start=1):
    print(f'{idx:02d}. {col}')

print('\nDtypes:')
display(df_raw.dtypes.to_frame('dtype'))

print('\nMissing values:')
display(df_raw.isna().sum().sort_values(ascending=False).to_frame('missing_count'))

print('\nSample rows:')
display(df_raw.head(5))
display(df_raw.tail(5))

In [ ]:
# Quick uniqueness and coordinate sanity checks.
inspect_df = df_raw.copy()
inspect_df['maps_latitude_num'] = pd.to_numeric(inspect_df.get('maps_latitude'), errors='coerce')
inspect_df['maps_longitude_num'] = pd.to_numeric(inspect_df.get('maps_longitude'), errors='coerce')

summary = {
    'rows': len(inspect_df),
    'unique_names': inspect_df['Tên địa điểm'].astype(str).str.strip().nunique(),
    'duplicate_name_rows': inspect_df.duplicated(subset=['Tên địa điểm']).sum(),
    'missing_latitude': inspect_df['maps_latitude_num'].isna().sum(),
    'missing_longitude': inspect_df['maps_longitude_num'].isna().sum(),
    'lat_outside_vietnam_bbox': (~inspect_df['maps_latitude_num'].between(7.0, 24.5)).sum(),
    'lng_outside_vietnam_bbox': (~inspect_df['maps_longitude_num'].between(102.0, 110.5)).sum(),
}
display(pd.Series(summary, name='value').to_frame())

print('Top destination types:')
display(df_raw['maps_destination_type'].fillna('Unknown').value_counts().head(20).to_frame('count'))

### A.2 Reusable preprocessing helpers

These helpers standardize Vietnamese text, parse ratings/reviews, clean keywords/hours, calculate distance, and define fuzzy membership curves.


In [ ]:
def strip_accents(value):
    text = unicodedata.normalize('NFD', str(value or ''))
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.replace('\u0111', 'd').replace('\u0110', 'D')


def normalize_text(value):
    text = strip_accents(value).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return ' '.join(text.split())


def clean_string(value):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else np.nan


def parse_rating(value):
    if pd.isna(value):
        return np.nan
    text = str(value).lower().replace(',', '.')
    match = re.search(r'(\d+(?:\.\d+)?)', text)
    if not match:
        return np.nan
    rating = float(match.group(1))
    if 0 <= rating <= 5:
        return rating
    if 0 <= rating <= 10:
        return rating / 2
    return np.nan


def parse_review_count(value):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('\xa0', ' ')
    match = re.search(r'([\d][\d,\.\s]*)', text)
    if not match:
        return np.nan
    digits = re.sub(r'\D', '', match.group(1))
    return float(digits) if digits else np.nan


def split_keywords(value):
    if pd.isna(value):
        return []
    text = str(value).replace('\u201c', '"').replace('\u201d', '"')
    parts = re.split(r'[,;|]', text)
    cleaned = []
    for part in parts:
        token = part.strip().strip('"').strip("'").strip()
        token = re.sub(r'\s+', ' ', token)
        if token:
            cleaned.append(token)
    return list(dict.fromkeys(cleaned))


def keywords_to_style(values):
    values = [str(v).strip() for v in values if str(v).strip()]
    values = list(dict.fromkeys(values))
    return ' , '.join(f'"{v}"' for v in values)


def infer_keywords(row):
    keywords = split_keywords(row.get('Từ Khóa'))
    type_text = normalize_text(row.get('maps_destination_type'))
    desc_text = normalize_text(row.get('Mô tả'))
    name_text = normalize_text(row.get('Tên địa điểm'))
    joined = ' '.join([type_text, desc_text, name_text])

    rules = [
        (['beach', 'bien', 'bai tam'], ['biển', 'nghỉ dưỡng', 'chụp ảnh']),
        (['museum', 'bao tang'], ['bảo tàng', 'văn hóa', 'tham quan']),
        (['temple', 'pagoda', 'chua', 'den', 'dinh', 'place worship'], ['tâm linh', 'văn hóa', 'tham quan']),
        (['viewpoint', 'ngam canh'], ['ngắm cảnh', 'chụp ảnh', 'tham quan']),
        (['theme park', 'amusement', 'zoo', 'aquarium'], ['vui chơi', 'giải trí', 'tham quan']),
        (['historic', 'heritage', 'di tich', 'thanh co'], ['di tích', 'lịch sử', 'tham quan']),
        (['park', 'garden', 'camp'], ['dã ngoại', 'vui chơi', 'tham quan']),
        (['market', 'cho'], ['chợ', 'văn hóa', 'trải nghiệm']),
    ]
    for needles, additions in rules:
        if any(needle in joined for needle in needles):
            keywords.extend(additions)

    if not keywords:
        keywords = ['tham quan', 'du lịch']
    return list(dict.fromkeys(keywords))


def clean_hours(value):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('\xa0', ' ')
    # Remove Google UI icons / repeated separators. Keep simple opening-hour strings.
    chunks = [c.strip() for c in text.split('|')]
    chunks = [c for c in chunks if c and not re.fullmatch(r'[\W_]+', c)]
    chunks = [c for c in chunks if 'busy at' not in c.lower()]
    if not chunks:
        return np.nan
    return ' | '.join(list(dict.fromkeys(chunks)))


def haversine_km(lat1, lon1, lat2, lon2):
    if any(pd.isna(v) for v in [lat1, lon1, lat2, lon2]):
        return np.nan
    radius = 6371.0088
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * radius * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def minmax(series):
    series = pd.to_numeric(series, errors='coerce')
    min_value = series.min()
    max_value = series.max()
    if pd.isna(min_value) or pd.isna(max_value) or min_value == max_value:
        return pd.Series(0.0, index=series.index)
    return (series - min_value) / (max_value - min_value)


def ramp_down(x, low, high):
    if pd.isna(x):
        return 0.0
    if x <= low:
        return 1.0
    if x >= high:
        return 0.0
    return float((high - x) / (high - low))


def fuzzy_text_score(query, text):
    query_norm = normalize_text(query)
    text_norm = normalize_text(text)
    if not query_norm or not text_norm:
        return 0.0
    return max(
        fuzz.ratio(query_norm, text_norm),
        fuzz.partial_ratio(query_norm, text_norm),
        fuzz.token_set_ratio(query_norm, text_norm),
    ) / 100

### A.3 Clean rows and fill missing values

Keep rows with valid names and Vietnam coordinates. Fill missing ratings, reviews, descriptions, keywords, and opening hours with transparent model-safe defaults.


In [ ]:
df = df_raw.copy()

# Standardize text columns.
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].map(clean_string)

# Parse numeric fields.
df['rating'] = df['Đánh giá '].map(parse_rating)
df['maps_review_count_clean'] = pd.to_numeric(df['maps_review_count'], errors='coerce')
missing_review_mask = df['maps_review_count_clean'].isna()
df.loc[missing_review_mask, 'maps_review_count_clean'] = df.loc[missing_review_mask, 'maps_review_label'].map(parse_review_count)

df['latitude'] = pd.to_numeric(df['maps_latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['maps_longitude'], errors='coerce')

# Drop rows that cannot be recommended geographically.
before_rows = len(df)
df = df.dropna(subset=['Tên địa điểm', 'latitude', 'longitude']).copy()
df = df[df['latitude'].between(7.0, 24.5) & df['longitude'].between(102.0, 110.5)].copy()
print('Dropped rows with missing/invalid core fields:', before_rows - len(df))

# Deduplicate near-identical POIs.
df['name_key'] = df['Tên địa điểm'].map(normalize_text)
df['coord_key'] = df['latitude'].round(5).astype(str) + ',' + df['longitude'].round(5).astype(str)
before_dedup = len(df)
df = df.drop_duplicates(subset=['name_key', 'coord_key']).copy()
print('Dropped duplicated name+coordinate rows:', before_dedup - len(df))

# Fill categorical/text fields.
df['poi_type'] = df['maps_destination_type'].fillna('Unknown').astype(str).str.strip()
df['result_name'] = df['maps_result_name'].fillna(df['Tên địa điểm'])
df['location'] = df['Vị trí'].fillna('Việt Nam')
df['description'] = df['Mô tả'].fillna('Điểm đến tại Việt Nam.')
df['image_url'] = df['Ảnh'].fillna('')
df['open_hours_clean'] = df['maps_open_hours'].fillna(df['maps_first_open_hours']).map(clean_hours)
df['open_hours_clean'] = df['open_hours_clean'].fillna('Không rõ')

# Fill keywords with same quoted style as the source workbook.
df['keyword_list'] = df.apply(infer_keywords, axis=1)
df['keywords_clean'] = df['keyword_list'].map(keywords_to_style)

# Fill ratings by type median, then global median.
global_rating_median = df['rating'].median()
type_rating_median = df.groupby('poi_type')['rating'].transform('median')
df['rating_filled'] = df['rating'].fillna(type_rating_median).fillna(global_rating_median).fillna(3.5)
df['rating_filled'] = df['rating_filled'].clip(0, 5)

# Review counts are popularity signals; missing reviews become 0 rather than imputed popularity.
df['review_count'] = df['maps_review_count_clean'].fillna(0).clip(lower=0)
df['has_image'] = (df['image_url'].astype(str).str.len() > 0).astype(int)
df['has_hours'] = (df['open_hours_clean'] != 'Không rõ').astype(int)

# Split criteria text to avoid double-counting.
# Content excludes type and location. Type/location are scored by separate criteria.
df['content_text'] = (
    df['Tên địa điểm'].fillna('') + ' ' +
    df['result_name'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['keyword_list'].map(lambda values: ' '.join(values))
)
df['content_text_norm'] = df['content_text'].map(normalize_text)

# Search text is only for direct fuzzy lookup, not for the content criterion.
df['search_text'] = (
    df['content_text'].fillna('') + ' ' +
    df['location'].fillna('') + ' ' +
    df['poi_type'].fillna('')
)
df['search_text_norm'] = df['search_text'].map(normalize_text)

# Keep source-style columns updated for cleaned export.
df['Đánh giá '] = df['rating_filled'].round(2)
df['Từ Khóa'] = df['keywords_clean']
df['maps_latitude'] = df['latitude']
df['maps_longitude'] = df['longitude']

print('Cleaned shape:', df.shape)
display(df[['Tên địa điểm', 'location', 'poi_type', 'rating', 'rating_filled', 'review_count', 'keywords_clean']].head(10))

### A.4 Normalize quality features

Normalize ratings, reviews, image availability, and opening-hours availability to comparable [0, 1] signals.


In [ ]:
df['rating_norm'] = (df['rating_filled'] / 5).clip(0, 1)
df['review_log'] = np.log1p(df['review_count'])
df['review_norm'] = minmax(df['review_log'])
df['quality_score'] = (
    0.55 * df['rating_norm'] +
    0.35 * df['review_norm'] +
    0.05 * df['has_image'] +
    0.05 * df['has_hours']
).clip(0, 1)

clean_report = pd.DataFrame({
    'column': ['rating_norm', 'review_norm', 'quality_score', 'has_image', 'has_hours'],
    'min': [df[c].min() for c in ['rating_norm', 'review_norm', 'quality_score', 'has_image', 'has_hours']],
    'max': [df[c].max() for c in ['rating_norm', 'review_norm', 'quality_score', 'has_image', 'has_hours']],
    'mean': [df[c].mean() for c in ['rating_norm', 'review_norm', 'quality_score', 'has_image', 'has_hours']],
})
display(clean_report)

print('Missing after preprocessing:')
display(df[['Tên địa điểm', 'location', 'description', 'keywords_clean', 'latitude', 'longitude', 'rating_filled', 'review_count']].isna().sum().to_frame('missing_count'))

### A.5 Build content text and TF-IDF

TF-IDF uses names, descriptions, and keywords. Type and location remain separate criteria to avoid double-counting.


In [ ]:
content_tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)
content_matrix = content_tfidf.fit_transform(df['content_text_norm'])

print('Content TF-IDF matrix shape:', content_matrix.shape)
print('Content criterion excludes `poi_type` and `location` to avoid double-counting.')

feature_preview = pd.DataFrame({
    'feature': content_tfidf.get_feature_names_out()[:30]
})
display(feature_preview.T)

## Phần B — Synthetic user behavior

The source workbook has no interaction log, so this section creates a deterministic event table with the requested schema. Event propensities use POI quality signals, but the generated table is only a development and evaluation scaffold.


In [ ]:
RNG = np.random.default_rng(42)
df = df.reset_index(drop=True)
df["poi_id"] = [f"POI-{i:05d}" for i in range(1, len(df) + 1)]
ACTION_WEIGHTS = {"view": 1.0, "click": 2.0, "save": 3.0, "like": 4.0, "visit": 5.0}
ACTION_NAMES = list(ACTION_WEIGHTS)
quality_prior = 0.35 + 0.65 * df["quality_score"].to_numpy(dtype=float)
quality_prior = quality_prior / quality_prior.sum()

events = []
start_time = pd.Timestamp("2025-01-01 08:00:00")
for user_number in range(1, 121):
    user_id = f"user_{user_number:04d}"
    n_events = int(RNG.integers(5, 16))
    chosen = RNG.choice(len(df), size=n_events, replace=False, p=quality_prior)
    offsets = np.sort(RNG.integers(0, 365 * 24 * 60, size=n_events))
    for index, offset in zip(chosen, offsets):
        events.append({
            "user_id": user_id,
            "poi_id": df.at[index, "poi_id"],
            "timestamp": start_time + pd.to_timedelta(int(offset), unit="m"),
            "action": RNG.choice(ACTION_NAMES, p=[0.48, 0.24, 0.12, 0.10, 0.06]),
        })
interactions = pd.DataFrame(events).sort_values(["timestamp", "user_id"]).reset_index(drop=True)
assert {"user_id", "poi_id", "timestamp", "action"}.issubset(interactions.columns)
print("Synthetic events:", interactions.shape)
display(interactions.head(10))


## Phần C — Behavioral recommender

The behavioral layer contains three models: action-weighted popularity, association rules from user transactions, and item-based collaborative filtering.


In [ ]:
def _behavior_events(events=None):
    table = interactions if events is None else events
    table = table.copy()
    table["event_weight"] = table["action"].map(ACTION_WEIGHTS).fillna(1.0)
    return table


def _scores_from_poi_series(values):
    values = pd.Series(values, dtype=float).reindex(df["poi_id"], fill_value=0.0)
    return pd.Series(minmax(values).to_numpy(), index=df.index).fillna(0.0)


def popularity_scores(events=None):
    table = _behavior_events(events)
    counts = table.groupby("poi_id")["event_weight"].sum() if not table.empty else pd.Series(dtype=float)
    return _scores_from_poi_series(counts)


def mine_association_rules(events=None, min_support=0.01, min_confidence=0.05):
    table = _behavior_events(events)
    if table.empty:
        return pd.DataFrame(columns=["antecedent", "consequent", "support", "confidence", "lift"])
    transactions = table.groupby("user_id")["poi_id"].apply(lambda values: set(values)).tolist()
    n_transactions = len(transactions)
    item_counts, pair_counts = {}, {}
    for transaction in transactions:
        for item in transaction:
            item_counts[item] = item_counts.get(item, 0) + 1
        for antecedent in transaction:
            for consequent in transaction:
                if antecedent != consequent:
                    pair_counts[(antecedent, consequent)] = pair_counts.get((antecedent, consequent), 0) + 1
    rows = []
    for (antecedent, consequent), pair_count in pair_counts.items():
        support = pair_count / n_transactions
        confidence = pair_count / item_counts[antecedent]
        consequent_support = item_counts[consequent] / n_transactions
        lift = confidence / consequent_support if consequent_support else 0.0
        if support >= min_support and confidence >= min_confidence:
            rows.append({"antecedent": antecedent, "consequent": consequent,
                         "support": support, "confidence": confidence, "lift": lift})
    return pd.DataFrame(rows).sort_values(["lift", "confidence"], ascending=False).reset_index(drop=True) if rows else pd.DataFrame(columns=["antecedent", "consequent", "support", "confidence", "lift"])


def association_rule_scores(user_id, events=None):
    table = _behavior_events(events)
    user_items = set(table.loc[table["user_id"].eq(user_id), "poi_id"])
    rules = mine_association_rules(events)
    scores = pd.Series(0.0, index=df["poi_id"])
    for row in rules[rules["antecedent"].isin(user_items)].itertuples(index=False):
        scores[row.consequent] += float(row.confidence * max(row.lift, 0.0))
    return _scores_from_poi_series(scores)


def item_cf_scores(user_id, events=None):
    table = _behavior_events(events)
    if table.empty:
        return popularity_scores(events)
    matrix = table.pivot_table(index="user_id", columns="poi_id", values="event_weight", aggfunc="sum", fill_value=0.0)
    matrix = matrix.reindex(columns=df["poi_id"], fill_value=0.0)
    if user_id not in matrix.index:
        return popularity_scores(events)
    user_vector = matrix.loc[user_id].to_numpy(dtype=float)
    seen_indices = np.flatnonzero(user_vector > 0)
    if len(seen_indices) == 0:
        return popularity_scores(events)
    # Compare every item only with this user's seen items; avoid an O(n_items^2) matrix.
    similarity_to_seen = cosine_similarity(matrix.T, matrix.T.iloc[seen_indices, :])
    raw = similarity_to_seen @ user_vector[seen_indices]
    raw[seen_indices] = 0.0
    return _scores_from_poi_series(pd.Series(raw, index=matrix.columns))

def behavior_model_scores(user_id=None, events=None):
    popularity = popularity_scores(events)
    if user_id is None:
        return popularity
    association = association_rule_scores(user_id, events)
    item_cf = item_cf_scores(user_id, events)
    return (0.30 * popularity + 0.25 * association + 0.45 * item_cf).clip(0, 1)


association_rules = mine_association_rules()
print("Association rules:", association_rules.shape)
display(association_rules.head(10))


## Phần D — Content model

TF-IDF and cosine similarity provide an interpretable model for cold-start users. The vectorizer and matrix are built in Phần A and reused here.


In [ ]:
def content_model_scores(profile=None):
    return pd.Series(content_similarity_scores(profile or {}), index=df.index).clip(0, 1)


def content_recommend(profile=None, top_k=10):
    result = df[["poi_id", "Tên địa điểm", "poi_type", "location", "quality_score"]].copy()
    result["content_score"] = content_model_scores(profile)
    return result.sort_values(["content_score", "quality_score"], ascending=False).head(top_k)


## Phần E — Hybrid

Existing users use behavior; new users use content. When both user_id and a profile are available, scores are blended on the same [0, 1] scale.


In [ ]:
def hybrid_scores(profile=None, user_id=None, behavior_weight=0.65):
    profile = profile or {}
    content = content_model_scores(profile)
    if user_id is None or not interactions["user_id"].eq(user_id).any():
        return content.rename("hybrid_score")
    behavior = behavior_model_scores(user_id)
    weight = float(np.clip(behavior_weight, 0, 1))
    return (weight * behavior + (1 - weight) * content).rename("hybrid_score")


def hybrid_recommend(profile=None, user_id=None, top_k=10):
    result = df[["poi_id", "Tên địa điểm", "poi_type", "location", "quality_score"]].copy()
    result["hybrid_score"] = hybrid_scores(profile, user_id).to_numpy()
    return result.sort_values(["hybrid_score", "quality_score"], ascending=False).head(top_k)


## Phần F — Contextual reranking

This stage combines behavioral and context signals before the final multi-criteria ranker.


### F.1 Shared criterion functions

These content, type, location, distance, and quality criteria are reused by contextual reranking and fuzzy AHP + TOPSIS.


In [ ]:
def content_similarity_scores(profile):
    query_parts = []
    profile = profile or {}
    for key in ['query', 'keywords']:
        value = profile.get(key, [])
        if isinstance(value, str):
            query_parts.append(value)
        else:
            query_parts.extend([str(item) for item in value])
    query_text = normalize_text(' '.join(query_parts))
    if not query_text:
        return np.zeros(len(df))
    query_vector = content_tfidf.transform([query_text])
    return cosine_similarity(query_vector, content_matrix).ravel()


def fuzzy_list_similarity(values, text_series):
    if isinstance(values, str):
        values = [values]
    values = [value for value in (values or []) if str(value).strip()]
    if not values:
        return np.zeros(len(text_series))

    scores = []
    for text in text_series.fillna('').astype(str):
        text_score = max(fuzzy_text_score(value, text) for value in values)
        scores.append(text_score)
    return np.array(scores)


def distance_scores(profile):
    lat = profile.get('current_lat') if profile else None
    lng = profile.get('current_lng') if profile else None
    max_distance_km = profile.get('max_distance_km', 300) if profile else 300
    if lat is None or lng is None:
        return np.ones(len(df)), np.full(len(df), np.nan)

    distances = np.array([
        haversine_km(lat, lng, row_lat, row_lng)
        for row_lat, row_lng in zip(df['latitude'], df['longitude'])
    ])
    scores = np.array([ramp_down(distance, 0, max_distance_km) for distance in distances])
    return scores, distances


def criteria_scores_for_profile(profile):
    profile = profile or {}
    content_scores = content_similarity_scores(profile)
    type_scores = fuzzy_list_similarity(profile.get('preferred_types'), df['poi_type'])
    location_scores = fuzzy_list_similarity(profile.get('preferred_locations'), df['location'])
    distance_score_values, distances = distance_scores(profile)
    quality = df['quality_score'].to_numpy()
    return pd.DataFrame({
        'content_criterion': content_scores,
        'type_criterion': type_scores,
        'location_criterion': location_scores,
        'distance_criterion': distance_score_values,
        'quality_criterion': quality,
        'distance_km': distances,
    }, index=df.index)

In [ ]:
def contextual_rerank(profile=None, user_id=None, top_k=10, weights=None):
    profile = profile or {}
    criteria = criteria_scores_for_profile(profile)
    context_weights = weights or {"behavior": 0.35, "distance": 0.20, "quality": 0.20, "type_context": 0.25}
    result = df.copy()
    result["behavior_score"] = behavior_model_scores(user_id).to_numpy()
    result["distance_score"] = criteria["distance_criterion"].to_numpy()
    result["quality_score"] = df["quality_score"].to_numpy()
    result["type_context_score"] = (
        0.60 * criteria["type_criterion"] +
        0.25 * criteria["location_criterion"] +
        0.15 * criteria["content_criterion"]
    ).to_numpy()
    result["contextual_score"] = (
        context_weights["behavior"] * result["behavior_score"] +
        context_weights["distance"] * result["distance_score"] +
        context_weights["quality"] * result["quality_score"] +
        context_weights["type_context"] * result["type_context_score"]
    )
    return result.sort_values(["contextual_score", "quality_score"], ascending=False).head(top_k)


## Phần G — Fuzzy AHP + TOPSIS

The ranking stage follows: criteria -> pairwise matrix -> consistency ratio (CR) -> fuzzy weights -> fuzzy evaluation -> H = A x W -> defuzzify -> TOPSIS -> Top-K.


In [ ]:
FUZZY_AHP_CRITERIA = ['content', 'type', 'location', 'distance', 'quality']

# Expert/user pairwise comparison input using Saaty's scale.
# Criteria order: content, type, location, distance, quality.
# Example interpretation:
# - content is moderately more important than location: 3
# - quality is equally important to content: 1
# - type and distance are equally important: 1
EXPERT_PAIRWISE_MATRIX = np.array([
    [1,   2,   3,   2,   1],
    [1/2, 1,   2,   1,   1/2],
    [1/3, 1/2, 1,   1/2, 1/3],
    [1/2, 1,   2,   1,   1/2],
    [1,   2,   3,   2,   1],
], dtype=float)

FUZZY_AHP_INFO = {}
RI_TABLE = {
    1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90, 5: 1.12,
    6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49,
}


def ahp_weights_from_pairwise(pairwise):
    matrix = np.asarray(pairwise, dtype=float)
    col_sum = matrix.sum(axis=0)
    normalized = matrix / col_sum
    weights = normalized.mean(axis=1)
    weights = weights / weights.sum()

    n = matrix.shape[0]
    weighted_sum = matrix @ weights
    consistency_vector = weighted_sum / weights
    lambda_max = consistency_vector.mean()
    ci = (lambda_max - n) / (n - 1) if n > 1 else 0.0
    ri = RI_TABLE.get(n, RI_TABLE[max(RI_TABLE)])
    cr = 0.0 if ri == 0 else ci / ri
    return weights, lambda_max, ci, cr


def triangular_from_ratio(value, uncertainty=0.20):
    value = float(value)
    if value <= 0:
        raise ValueError('Pairwise comparison values must be positive.')
    return np.array([
        value * (1 - uncertainty),
        value,
        value * (1 + uncertainty),
    ], dtype=float)


def fuzzy_pairwise_from_crisp(pairwise, uncertainty=0.20):
    matrix = np.asarray(pairwise, dtype=float)
    n = matrix.shape[0]
    fuzzy_matrix = np.zeros((n, n, 3), dtype=float)
    for i in range(n):
        for j in range(n):
            if i == j:
                fuzzy_matrix[i, j] = [1.0, 1.0, 1.0]
            elif i < j:
                fuzzy_matrix[i, j] = triangular_from_ratio(matrix[i, j], uncertainty)
                l, m, u = fuzzy_matrix[i, j]
                fuzzy_matrix[j, i] = [1 / u, 1 / m, 1 / l]
    return fuzzy_matrix


def fuzzy_weights_from_pairwise(fuzzy_matrix):
    # Buckley-style fuzzy geometric mean followed by fuzzy normalization.
    n = fuzzy_matrix.shape[0]
    geometric_means = np.zeros((n, 3), dtype=float)
    for i in range(n):
        row_product = np.prod(fuzzy_matrix[i], axis=0)
        geometric_means[i] = row_product ** (1 / n)

    sum_l = geometric_means[:, 0].sum()
    sum_m = geometric_means[:, 1].sum()
    sum_u = geometric_means[:, 2].sum()
    fuzzy_weights = np.column_stack([
        geometric_means[:, 0] / sum_u,
        geometric_means[:, 1] / sum_m,
        geometric_means[:, 2] / sum_l,
    ])
    defuzzified = fuzzy_weights.mean(axis=1)
    defuzzified = defuzzified / defuzzified.sum()
    return fuzzy_weights, defuzzified


def score_to_triangular(score, spread=0.08):
    score = float(np.clip(score, 0, 1))
    return np.array([
        max(0.0, score - spread),
        score,
        min(1.0, score + spread),
    ])


def fuzzy_ahp_scores(profile, pairwise_matrix=None, uncertainty=0.20, eval_spread=0.08):
    pairwise = np.asarray(pairwise_matrix if pairwise_matrix is not None else EXPERT_PAIRWISE_MATRIX, dtype=float)
    crisp_weights, lambda_max, ci, cr = ahp_weights_from_pairwise(pairwise)
    fuzzy_pairwise = fuzzy_pairwise_from_crisp(pairwise, uncertainty=uncertainty)
    fuzzy_weights, defuzz_weights = fuzzy_weights_from_pairwise(fuzzy_pairwise)

    criteria_df = criteria_scores_for_profile(profile)
    criteria_cols = [
        'content_criterion', 'type_criterion', 'location_criterion',
        'distance_criterion', 'quality_criterion'
    ]
    x = criteria_df[criteria_cols].to_numpy(dtype=float)

    fuzzy_eval = np.zeros((len(criteria_df), len(criteria_cols), 3), dtype=float)
    for j in range(len(criteria_cols)):
        fuzzy_eval[:, j, :] = np.vstack([score_to_triangular(value, eval_spread) for value in x[:, j]])

    # H = A x W in triangular fuzzy form.
    fuzzy_h = np.zeros((len(criteria_df), 3), dtype=float)
    for j in range(len(criteria_cols)):
        fuzzy_h += fuzzy_eval[:, j, :] * fuzzy_weights[j]

    # Defuzzification by centroid, then normalization for ranking.
    fuzzy_ahp_score = fuzzy_h.mean(axis=1)
    fuzzy_ahp_norm = minmax(pd.Series(fuzzy_ahp_score, index=criteria_df.index)).to_numpy()

    FUZZY_AHP_INFO.clear()
    FUZZY_AHP_INFO.update({
        'criteria': FUZZY_AHP_CRITERIA,
        'pairwise_matrix': pairwise,
        'crisp_weights': crisp_weights,
        'fuzzy_weights': fuzzy_weights,
        'defuzzified_weights': defuzz_weights,
        'lambda_max': lambda_max,
        'ci': ci,
        'cr': cr,
        'cr_accepted': cr <= 0.10,
    })

    return criteria_df.join(pd.DataFrame({
        'fuzzy_ahp_score': fuzzy_ahp_score,
        'fuzzy_ahp_norm': fuzzy_ahp_norm,
        'final_score': fuzzy_ahp_norm,
    }, index=df.index))


fuzzy_ahp_scores({})
weights_table = pd.DataFrame({
    'criterion': FUZZY_AHP_INFO['criteria'],
    'ahp_weight_from_pairwise': FUZZY_AHP_INFO['crisp_weights'],
    'fuzzy_weight_l': FUZZY_AHP_INFO['fuzzy_weights'][:, 0],
    'fuzzy_weight_m': FUZZY_AHP_INFO['fuzzy_weights'][:, 1],
    'fuzzy_weight_u': FUZZY_AHP_INFO['fuzzy_weights'][:, 2],
    'defuzzified_weight': FUZZY_AHP_INFO['defuzzified_weights'],
})
print('AHP Consistency Ratio:', round(FUZZY_AHP_INFO['cr'], 6), '| accepted:', FUZZY_AHP_INFO['cr_accepted'])
display(weights_table)

In [ ]:
TOPSIS_CRITERIA = ["behavior", "content", "distance", "quality", "context"]


def fuzzy_ahp_topsis(
    profile=None,
    user_id=None,
    candidate_indices=None,
    candidate_pool_size=200,
    pairwise_matrix=None,
    uncertainty=0.20,
    eval_spread=0.08,
):
    """Generate candidates, contextualize them, then rank with fuzzy AHP and TOPSIS."""
    profile = profile or {}
    known_user = user_id is not None and interactions["user_id"].eq(user_id).any()
    content_scores = content_model_scores(profile)
    behavior_scores = behavior_model_scores(user_id) if known_user else pd.Series(0.0, index=df.index)

    # Candidate generation: behavior/content hybrid for existing users, content for cold start.
    candidate_scores = hybrid_scores(profile, user_id)
    if candidate_indices is None:
        candidate_indices = candidate_scores.sort_values(ascending=False).head(
            min(max(int(candidate_pool_size), 1), len(df))
        ).index
    else:
        candidate_indices = pd.Index(candidate_indices).intersection(df.index)
    if len(candidate_indices) == 0:
        candidate_indices = candidate_scores.sort_values(ascending=False).head(1).index

    criteria = criteria_scores_for_profile(profile)
    type_context = (
        0.60 * criteria["type_criterion"] +
        0.25 * criteria["location_criterion"] +
        0.15 * criteria["content_criterion"]
    )
    decision = pd.DataFrame({
        "candidate_score": candidate_scores,
        "behavior_score": behavior_scores,
        "content_score": content_scores,
        "distance_score": criteria["distance_criterion"],
        "quality_score": df["quality_score"],
        "type_context_score": type_context,
        "contextual_score": (
            0.35 * behavior_scores +
            0.20 * criteria["distance_criterion"] +
            0.20 * df["quality_score"] +
            0.25 * type_context
        ),
        "distance_km": criteria["distance_km"],
    }, index=df.index).loc[candidate_indices]

    pairwise = np.asarray(
        pairwise_matrix if pairwise_matrix is not None else EXPERT_PAIRWISE_MATRIX,
        dtype=float,
    )
    if pairwise.shape != (len(TOPSIS_CRITERIA), len(TOPSIS_CRITERIA)):
        raise ValueError(f"pairwise_matrix must be {len(TOPSIS_CRITERIA)}x{len(TOPSIS_CRITERIA)}.")
    crisp_weights, lambda_max, ci, cr = ahp_weights_from_pairwise(pairwise)
    fuzzy_pairwise = fuzzy_pairwise_from_crisp(pairwise, uncertainty=uncertainty)
    fuzzy_weights, defuzz_weights = fuzzy_weights_from_pairwise(fuzzy_pairwise)

    criteria_cols = ["behavior_score", "content_score", "distance_score", "quality_score", "type_context_score"]
    raw = decision[criteria_cols].to_numpy(dtype=float)
    fuzzy_eval = np.zeros((len(decision), len(criteria_cols), 3), dtype=float)
    for j in range(len(criteria_cols)):
        fuzzy_eval[:, j, :] = np.vstack([
            score_to_triangular(value, eval_spread) for value in raw[:, j]
        ])

    # H = A x W: weighted fuzzy decision matrix, then defuzzify each criterion.
    fuzzy_h = fuzzy_eval * fuzzy_weights[None, :, :]
    h_defuzzified = fuzzy_h.mean(axis=2)
    vector_norm = np.sqrt((h_defuzzified ** 2).sum(axis=0))
    vector_norm[vector_norm == 0] = 1.0
    normalized = h_defuzzified / vector_norm
    weighted = normalized * defuzz_weights[None, :]
    ideal_best, ideal_worst = weighted.max(axis=0), weighted.min(axis=0)
    distance_best = np.sqrt(((weighted - ideal_best) ** 2).sum(axis=1))
    distance_worst = np.sqrt(((weighted - ideal_worst) ** 2).sum(axis=1))
    topsis_score = np.nan_to_num(
        distance_worst / (distance_best + distance_worst),
        nan=0.0,
    )

    FUZZY_AHP_INFO.clear()
    FUZZY_AHP_INFO.update({
        "criteria": TOPSIS_CRITERIA,
        "pairwise_matrix": pairwise,
        "crisp_weights": crisp_weights,
        "fuzzy_weights": fuzzy_weights,
        "defuzzified_weights": defuzz_weights,
        "lambda_max": lambda_max,
        "ci": ci,
        "cr": cr,
        "cr_accepted": cr <= 0.10,
        "candidate_count": len(decision),
        "known_user": known_user,
        "H_defuzzified": h_defuzzified,
        "topsis_ideal_best": ideal_best,
        "topsis_ideal_worst": ideal_worst,
    })

    # Keep the exported weight table aligned with the final production ranker.
    global weights_table
    weights_table = pd.DataFrame({
        "criterion": TOPSIS_CRITERIA,
        "ahp_weight_from_pairwise": crisp_weights,
        "fuzzy_weight_l": fuzzy_weights[:, 0],
        "fuzzy_weight_m": fuzzy_weights[:, 1],
        "fuzzy_weight_u": fuzzy_weights[:, 2],
        "defuzzified_weight": defuzz_weights,
    })

    return decision.drop(columns=["quality_score"]).join(pd.DataFrame({
        "fuzzy_h_score": h_defuzzified.sum(axis=1),
        "topsis_score": topsis_score,
        "final_score": topsis_score,
    }, index=decision.index))


### G.1 Top-K recommendation API

The public recommendation function now follows the production flow:
candidate generation -> behavior/content scoring -> contextual signals -> Fuzzy AHP -> TOPSIS -> Top-K.


In [ ]:
def recommend_pois(
    profile=None,
    user_id=None,
    top_k=20,
    exclude_names=None,
    min_final_score=0.0,
    pairwise_matrix=None,
    candidate_pool_size=None,
):
    profile = profile or {}
    exclude_names = {normalize_text(name) for name in (exclude_names or [])}
    pool_size = candidate_pool_size or max(100, top_k * 10)

    # Candidate generation -> contextual signals -> fuzzy AHP -> TOPSIS -> Top-K.
    ranked_scores = fuzzy_ahp_topsis(
        profile=profile,
        user_id=user_id,
        candidate_pool_size=pool_size,
        pairwise_matrix=pairwise_matrix,
    )
    result = df.join(ranked_scores, how="inner")
    if exclude_names:
        result = result[~result["name_key"].isin(exclude_names)]
    result = result[result["final_score"] >= min_final_score]

    output_cols = [
        "poi_id", "Tên địa điểm", "location", "poi_type", "rating_filled", "review_count",
        "latitude", "longitude", "distance_km", "keywords_clean", "open_hours_clean",
        "maps_url", "candidate_score", "behavior_score", "content_score",
        "distance_score", "quality_score", "type_context_score", "contextual_score",
        "fuzzy_h_score", "topsis_score", "final_score",
    ]
    ranked = result.sort_values(
        ["final_score", "contextual_score", "quality_score"],
        ascending=False,
    )[output_cols].head(top_k).copy()
    ranked["distance_km"] = ranked["distance_km"].round(1)
    for col in [
        "candidate_score", "behavior_score", "content_score", "distance_score",
        "quality_score", "type_context_score", "contextual_score", "fuzzy_h_score",
        "topsis_score", "final_score",
    ]:
        ranked[col] = ranked[col].round(4)
    return ranked


def fuzzy_search_poi(query, top_k=10):
    query_norm = normalize_text(query)
    if not query_norm:
        return pd.DataFrame()

    result = df.copy()
    result["fuzzy_name_score"] = result["Tên địa điểm"].map(
        lambda value: fuzzy_text_score(query_norm, value)
    )
    result["fuzzy_full_text_score"] = result["search_text_norm"].map(
        lambda value: fuzzy_text_score(query_norm, value)
    )
    result["fuzzy_search_score"] = np.maximum(
        result["fuzzy_name_score"], result["fuzzy_full_text_score"]
    )
    cols = [
        "Tên địa điểm", "location", "poi_type", "rating_filled", "review_count",
        "maps_url", "fuzzy_name_score", "fuzzy_full_text_score", "fuzzy_search_score",
    ]
    return result.sort_values("fuzzy_search_score", ascending=False)[cols].head(top_k)


## Examples

The examples exercise content, contextual, hybrid, and TOPSIS paths with realistic Vietnam travel profiles.


In [ ]:
# Example: cultural + scenic destinations near Nha Trang / Khanh Hoa.
user_profile = {
    'query': 'di tích văn hóa chụp ảnh ngắm cảnh',
    'keywords': ['văn hóa', 'di tích', 'chụp ảnh', 'ngắm cảnh'],
    'preferred_types': ['Tourist attraction', 'Historical landmark', 'Museum', 'Temple', 'Beach'],
    'preferred_locations': ['Khánh Hòa', 'Nha Trang'],
    'current_lat': 12.2388,
    'current_lng': 109.1967,
    'max_distance_km': 120,
}

recommendations = recommend_pois(user_profile, top_k=15)
display(recommendations)

In [ ]:
# Example: fuzzy search handles misspellings and partial user input.
display(fuzzy_search_poi('thap ba ponaga', top_k=10))
display(fuzzy_search_poi('vin wonder nha trang', top_k=10))

In [ ]:
# Example: beach/leisure recommendation around Da Nang.
da_nang_profile = {
    'query': 'biển vui chơi nghỉ dưỡng chụp ảnh',
    'keywords': ['biển', 'vui chơi', 'nghỉ dưỡng', 'chụp ảnh'],
    'preferred_types': ['Beach', 'Tourist attraction', 'Park'],
    'preferred_locations': ['Đà Nẵng', 'Hội An', 'Quảng Nam'],
    'current_lat': 16.0471,
    'current_lng': 108.2062,
    'max_distance_km': 80,
}

display(recommend_pois(da_nang_profile, top_k=15))

## Phần H — Evaluation

The original profile-based weak-label comparison is retained as a diagnostic baseline. A chronological event holdout with behavioral metrics is added below it.


In [ ]:
EVAL_K = 10

EVAL_PROFILES = [
    {
        'name': 'culture_nha_trang',
        'query': 'culture heritage museum temple viewpoint photo',
        'keywords': ['culture', 'heritage', 'museum', 'temple', 'viewpoint'],
        'preferred_types': ['Tourist attraction', 'Historical landmark', 'Museum', 'Temple', 'Buddhist temple'],
        'preferred_locations': ['Khanh Hoa', 'Nha Trang'],
        'current_lat': 12.2388,
        'current_lng': 109.1967,
        'max_distance_km': 120,
    },
    {
        'name': 'beach_da_nang',
        'query': 'beach leisure photo scenic park',
        'keywords': ['beach', 'leisure', 'photo', 'scenic'],
        'preferred_types': ['Beach', 'Tourist attraction', 'Park', 'Scenic spot'],
        'preferred_locations': ['Da Nang', 'Hoi An', 'Quang Nam'],
        'current_lat': 16.0471,
        'current_lng': 108.2062,
        'max_distance_km': 90,
    },
    {
        'name': 'spiritual_hanoi',
        'query': 'temple pagoda church spiritual culture',
        'keywords': ['temple', 'pagoda', 'church', 'spiritual', 'culture'],
        'preferred_types': ['Buddhist temple', 'Catholic church', 'Place of worship', 'Pagoda', 'Shrine'],
        'preferred_locations': ['Ha Noi', 'Hanoi'],
        'current_lat': 21.0278,
        'current_lng': 105.8342,
        'max_distance_km': 80,
    },
    {
        'name': 'nature_mountain_north',
        'query': 'mountain viewpoint waterfall lake national park',
        'keywords': ['mountain', 'viewpoint', 'waterfall', 'lake', 'national park'],
        'preferred_types': ['Scenic spot', 'Viewpoint', 'National park', 'Lake', 'Tourist attraction'],
        'preferred_locations': ['Lao Cai', 'Ha Giang', 'Ninh Binh'],
        'current_lat': 22.3364,
        'current_lng': 103.8438,
        'max_distance_km': 250,
    },
]


def pseudo_relevance(profile):
    criteria = criteria_scores_for_profile(profile)
    oracle = (
        0.25 * criteria['content_criterion'] +
        0.15 * criteria['type_criterion'] +
        0.15 * criteria['location_criterion'] +
        0.20 * criteria['distance_criterion'] +
        0.25 * criteria['quality_criterion']
    )
    threshold = oracle.quantile(0.95)
    relevance = (oracle >= threshold).astype(int)
    if relevance.sum() < EVAL_K:
        relevance.loc[oracle.sort_values(ascending=False).head(EVAL_K).index] = 1
    return oracle, relevance


def precision_at_k(relevance_ranked, k):
    values = np.asarray(relevance_ranked[:k], dtype=float)
    return values.mean() if len(values) else 0.0


def recall_at_k(relevance_ranked, total_relevant, k):
    if total_relevant == 0:
        return 0.0
    return float(np.asarray(relevance_ranked[:k]).sum() / total_relevant)


def average_precision_at_k(relevance_ranked, k):
    values = np.asarray(relevance_ranked[:k], dtype=float)
    hits = 0
    precisions = []
    for idx, rel in enumerate(values, start=1):
        if rel > 0:
            hits += 1
            precisions.append(hits / idx)
    return float(np.mean(precisions)) if precisions else 0.0


def reciprocal_rank_at_k(relevance_ranked, k):
    for idx, rel in enumerate(relevance_ranked[:k], start=1):
        if rel > 0:
            return 1 / idx
    return 0.0


def ndcg_at_k(relevance_ranked, relevance_scores_ranked, k):
    gains = np.asarray(relevance_scores_ranked[:k], dtype=float) * np.asarray(relevance_ranked[:k], dtype=float)
    discounts = 1 / np.log2(np.arange(2, len(gains) + 2))
    dcg = float(np.sum(gains * discounts))
    ideal_gains = np.sort(gains)[::-1]
    idcg = float(np.sum(ideal_gains * discounts))
    return dcg / idcg if idcg > 0 else 0.0


def ranking_diversity(indices):
    subset = df.loc[indices]
    type_diversity = subset['poi_type'].nunique() / max(1, len(subset))
    return float(type_diversity)


def crisp_ahp_weighted_scores(profile):
    criteria = criteria_scores_for_profile(profile)
    weights, _, _, _ = ahp_weights_from_pairwise(EXPERT_PAIRWISE_MATRIX)
    cols = ['content_criterion', 'type_criterion', 'location_criterion', 'distance_criterion', 'quality_criterion']
    return pd.Series(criteria[cols].to_numpy(dtype=float) @ weights, index=df.index)


def model_scores(profile):
    criteria = criteria_scores_for_profile(profile)
    main = fuzzy_ahp_scores(profile)['final_score']
    return {
        'popularity_baseline': criteria['quality_criterion'],
        'distance_baseline': criteria['distance_criterion'],
        'content_baseline': criteria['content_criterion'],
        'type_location_baseline': 0.50 * criteria['type_criterion'] + 0.50 * criteria['location_criterion'],
        'equal_weight_baseline': criteria[[
            'content_criterion', 'type_criterion', 'location_criterion',
            'distance_criterion', 'quality_criterion'
        ]].mean(axis=1),
        'crisp_ahp_weighted_baseline': crisp_ahp_weighted_scores(profile),
        'fuzzy_ahp': main,
    }


def evaluate_models(profiles, k=10):
    rows = []
    coverage_tracker = {}
    for profile in profiles:
        oracle, relevance = pseudo_relevance(profile)
        total_relevant = int(relevance.sum())
        scores_by_model = model_scores(profile)
        for model_name, scores in scores_by_model.items():
            ranked_indices = scores.sort_values(ascending=False).head(k).index
            rel_ranked = relevance.loc[ranked_indices].to_numpy()
            oracle_ranked = oracle.loc[ranked_indices].to_numpy()
            coverage_tracker.setdefault(model_name, set()).update(ranked_indices.tolist())
            rows.append({
                'profile': profile['name'],
                'model': model_name,
                f'precision@{k}': precision_at_k(rel_ranked, k),
                f'recall@{k}': recall_at_k(rel_ranked, total_relevant, k),
                f'map@{k}': average_precision_at_k(rel_ranked, k),
                f'mrr@{k}': reciprocal_rank_at_k(rel_ranked, k),
                f'ndcg@{k}': ndcg_at_k(rel_ranked, oracle_ranked, k),
                f'type_diversity@{k}': ranking_diversity(ranked_indices),
            })
    metrics = pd.DataFrame(rows)
    summary = metrics.groupby('model').mean(numeric_only=True).reset_index()
    summary[f'catalog_coverage@{k}'] = summary['model'].map(lambda model: len(coverage_tracker.get(model, set())) / len(df))
    return metrics, summary.sort_values(f'ndcg@{k}', ascending=False)


evaluation_by_profile, evaluation_summary = evaluate_models(EVAL_PROFILES, EVAL_K)
display(evaluation_by_profile)
display(evaluation_summary)

### H.1 Chronological train/test evaluation

For each user, the last event is held out as test data and all earlier events are used for training. The metrics use the requested names and include catalog coverage and POI-type diversity.


In [ ]:
def chronological_split(events):
    ordered = events.sort_values(["user_id", "timestamp"])
    trains, tests = [], []
    for _, user_events in ordered.groupby("user_id", sort=False):
        if len(user_events) >= 2:
            trains.append(user_events.iloc[:-1])
            tests.append(user_events.iloc[-1:])
    return (pd.concat(trains, ignore_index=True), pd.concat(tests, ignore_index=True))


def _hit(relevant, ranked, k):
    return float(relevant in list(ranked)[:k])


def _mrr(relevant, ranked, k):
    for pos, poi_id in enumerate(list(ranked)[:k], start=1):
        if poi_id == relevant:
            return 1.0 / pos
    return 0.0


def _ndcg(relevant, ranked, k):
    for pos, poi_id in enumerate(list(ranked)[:k], start=1):
        if poi_id == relevant:
            return 1.0 / np.log2(pos + 1)
    return 0.0


def _diversity(ranked, k):
    types = df.set_index("poi_id").reindex(list(ranked)[:k])["poi_type"].dropna()
    return float(types.nunique() / max(1, len(types)))


def evaluate_chronological(events, k=10):
    train, test = chronological_split(events)
    models = {
        "popularity": lambda user, profile: popularity_scores(train),
        "association_rules": lambda user, profile: association_rule_scores(user, train),
        "item_based_cf": lambda user, profile: item_cf_scores(user, train),
        "behavioral_hybrid": lambda user, profile: behavior_model_scores(user, train),
        "content_cold_start": lambda user, profile: content_model_scores(profile),
    }
    rows, coverage = [], {}
    for held_out in test.itertuples(index=False):
        history = set(train.loc[train["user_id"].eq(held_out.user_id), "poi_id"])
        poi = df.loc[df["poi_id"].eq(held_out.poi_id)].iloc[0]
        profile = {"query": poi["content_text"], "preferred_types": [poi["poi_type"]], "preferred_locations": [poi["location"]]}
        for name, scorer in models.items():
            scores = scorer(held_out.user_id, profile)
            allowed = scores.drop(index=df.index[df["poi_id"].isin(history)], errors="ignore")
            ranked_idx = allowed.sort_values(ascending=False).head(k).index
            ranked = df.loc[ranked_idx, "poi_id"].tolist()
            coverage.setdefault(name, set()).update(ranked)
            rows.append({
                "user_id": held_out.user_id, "model": name,
                f"HitRate@{k}": _hit(held_out.poi_id, ranked, k),
                f"Recall@{k}": _hit(held_out.poi_id, ranked, k),
                f"MRR@{k}": _mrr(held_out.poi_id, ranked, k),
                f"NDCG@{k}": _ndcg(held_out.poi_id, ranked, k),
                f"Diversity@{k}": _diversity(ranked, k),
            })
    by_user = pd.DataFrame(rows)
    summary = by_user.groupby("model").mean(numeric_only=True).reset_index()
    summary[f"Coverage@{k}"] = summary["model"].map(lambda name: len(coverage.get(name, set())) / max(1, len(df)))
    return train, test, by_user, summary.sort_values(f"HitRate@{k}", ascending=False)


behavior_train, behavior_test, chronological_metrics, chronological_summary = evaluate_chronological(interactions, 10)
display(chronological_summary)


## Outputs

Save the cleaned POI catalog, synthetic behavior table, recommendations, evaluation metrics, and fuzzy AHP weights.


In [ ]:
export_cols = [
    'STT', 'Tên địa điểm', 'location', 'description', 'rating_filled', 'review_count',
    'image_url', 'keywords_clean', 'poi_type', 'latitude', 'longitude', 'open_hours_clean',
    'quality_score', 'content_text_norm', 'search_text_norm', 'maps_url'
]
clean_export = df[export_cols].copy()
clean_export = clean_export.rename(columns={
    'location': 'Vị trí_clean',
    'description': 'Mô tả_clean',
    'rating_filled': 'rating_clean',
    'review_count': 'review_count_clean',
    'image_url': 'Ảnh_clean',
    'keywords_clean': 'Từ Khóa_clean',
    'poi_type': 'destination_type_clean',
    'latitude': 'lat_clean',
    'longitude': 'lng_clean',
})

clean_export.to_excel(CLEAN_OUTPUT_PATH, index=False)
recommendations.to_excel(SAMPLE_RECOMMENDATIONS_PATH, index=False)
with pd.ExcelWriter(EVALUATION_OUTPUT_PATH) as writer:
    evaluation_summary.to_excel(writer, sheet_name='summary', index=False)
    evaluation_by_profile.to_excel(writer, sheet_name='by_profile', index=False)
    weights_table.to_excel(writer, sheet_name='fuzzy_ahp_weights', index=False)

print('Saved cleaned recommendation data:', CLEAN_OUTPUT_PATH)
print('Saved sample recommendations:', SAMPLE_RECOMMENDATIONS_PATH)
print('Saved evaluation metrics:', EVALUATION_OUTPUT_PATH)
print('Clean export shape:', clean_export.shape)
interactions.to_excel("data/synthetic_user_behavior.xlsx", index=False)
chronological_metrics.to_excel("data/chronological_metrics.xlsx", index=False)
chronological_summary.to_excel("data/chronological_summary.xlsx", index=False)
print("Saved synthetic behavior and chronological evaluation artifacts.")


## Notes

- Edit EXPERT_PAIRWISE_MATRIX to represent expert/user pairwise judgments.
- Inspect CR <= 0.10 before using fuzzy AHP weights.
- Use content scores for cold-start users and behavior scores only when historical events exist.
- The synthetic behavior table is a scaffold for replacing weak labels with real interaction logs.
